# 2D Optimisation Explorer

Interactive browser for `results/2d_opt` plots and pkl data.

In [ ]:
import os, pickle, glob
import numpy as np
import matplotlib.pyplot as plt
from pypscan.jupyter import JupyterPScan

## PDF browser

Parametric widgets let you filter by N, optimizer, lr, loss, parameter pair, and track.

In [ ]:
from pypscan.jupyter import JupyterPScan

# Filename pattern:
#   2d_opt_N{N}_{optimizer}_lr{lr[_cosine]}_{loss}_{pair}_{track}.pdf
# (6 old files without lr are excluded — they predate the lr-in-filename convention)
# loss options ordered longest-first so regex alternation matches correctly
browser = JupyterPScan(
    regex=(
        r"2d_opt_N(?P<N>\d+)"
        r"_(?P<optimizer>[^_]+)"
        r"_lr(?P<lr>[0-9.]+(?:_cosine)?)"
        r"_(?P<loss>sobolev_loss_geomean_log1p|sobolev_loss|mse_loss)"
        r"_(?P<pair>.+\+.+)"
        r"_(?P<track>[^_+.]+)\.pdf"
    ),
    base_path="../results/2d_opt/",
)
browser.run()

## Load pkl files

Load all matching pkl files into a list of result dicts for further analysis.

In [ ]:
results = []
for path in sorted(glob.glob("../results/2d_opt/*.pkl")):
    with open(path, "rb") as f:
        r = pickle.load(f)
    r["_path"] = path
    results.append(r)

print(f"Loaded {len(results)} pkl files")
print("Keys:", list(results[0].keys()))

In [ ]:
# Summary table
print(f"{'loss':<35} {'pair':<45} {'optimizer':<12} {'lr':<10} {'N':>4}")
print("-" * 110)
for r in results:
    pair = "+".join(r["param_names"])
    sched = "_cosine" if r.get("lr_schedule") == "cosine" else ""
    print(f"{r['loss_name']:<35} {pair:<45} {r['optimizer']:<12} {str(r['lr'])+sched:<10} {r['N']:>4}")

## Final parameter values

For each result, plot the distribution of final p_n values across trials — how close do they land to GT?

In [ ]:
# Pick a result to inspect
r = results[0]  # change index as needed

p1, p2   = r["param_names"]
pn_gt1, pn_gt2 = r["p_n_gts"]
pair_tag = f"{p1} + {p2}"

final_pn = np.array([t["param_trajectory"][-1] for t in r["trials"]])
losses   = np.array([t["loss_trajectory"][-1]  for t in r["trials"]])

fig, axes = plt.subplots(1, 3, figsize=(14, 4))

axes[0].scatter(final_pn[:, 0], final_pn[:, 1], c=np.log10(losses + 1e-30),
                cmap="viridis", s=40, zorder=2)
axes[0].plot(pn_gt1, pn_gt2, "r*", ms=14, label="GT", zorder=3)
axes[0].set_xlabel(f"p_n  {p1}"); axes[0].set_ylabel(f"p_n  {p2}")
axes[0].set_title("Final p_n (colour = log loss)")
axes[0].legend(); axes[0].grid(True, alpha=0.3)

axes[1].hist(final_pn[:, 0] - pn_gt1, bins=20)
axes[1].axvline(0, color="red", ls="--")
axes[1].set_xlabel(f"Δp_n  {p1}"); axes[1].set_title(f"Error in {p1}")
axes[1].grid(True, alpha=0.3)

axes[2].hist(final_pn[:, 1] - pn_gt2, bins=20)
axes[2].axvline(0, color="red", ls="--")
axes[2].set_xlabel(f"Δp_n  {p2}"); axes[2].set_title(f"Error in {p2}")
axes[2].grid(True, alpha=0.3)

fig.suptitle(f"{pair_tag}  |  {r['loss_name']}  |  {r['optimizer']}  lr={r['lr']}  N={r['N']}",
             fontsize=10)
plt.tight_layout()
plt.show()

## Loss trajectories

Plot all trial loss trajectories for a chosen result.

In [ ]:
r = results[0]  # change index as needed

fig, ax = plt.subplots(figsize=(9, 4))
for trial in r["trials"]:
    ax.plot(trial["loss_trajectory"], lw=0.8, alpha=0.6)
ax.set_yscale("log")
ax.set_xlabel("step"); ax.set_ylabel("loss  (log)")
ax.set_title(f"{'+'.join(r['param_names'])}  |  {r['loss_name']}  |  {r['optimizer']}  lr={r['lr']}")
ax.grid(True, which="both", alpha=0.25)
plt.tight_layout()
plt.show()